**Setup & Loading**

In [ ]:
! pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=116fed2e436b68b80f15d84b807bfba3c0c823c4c42a1039eae5c5bf8aa1d6f2
  Stored in directory: /root/.cache/pip/wheels/7c/04/5c/157dc9106512a6c7a30653ec064490c94a49e0fc8f63d19ab9
Successfully built lime


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix, roc_auc_score, roc_curve,
                              classification_report, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
import shap
import lime
import lime.lime_tabular
import warnings
warnings.filterwarnings("ignore")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os

folder_path = '/content/drive/MyDrive/FM Data'

files = os.listdir(folder_path)
print(files)


In [ ]:
folder_path = "/content/drive/MyDrive/FM Data/FM12/"

train = pd.read_csv(folder_path + "training.csv")
oos   = pd.read_csv(folder_path + "OOS test.csv")
oot   = pd.read_csv(folder_path + "OOT test.csv")

print(train.shape, oos.shape, oot.shape)
print(train['DFlag'].value_counts(normalize=True))

**Exploratory Data Analysis**

In [ ]:
# Basic info
print(train.info())
print(train.describe())

# Missing values
print("Missing values per column:\n", train.isnull().sum()[train.isnull().sum() > 0])


In [ ]:
# Class distribution
plt.figure(figsize=(5,4))
sns.countplot(x='DFlag', data=train)
plt.title("Class Distribution: Default (1) vs Non-Default (0)")
counts = train['DFlag'].value_counts()
plt.text(0, counts[0]+2000, str(counts[0]), ha='center')
plt.text(1, counts[1]+2000, str(counts[1]), ha='center')
plt.show()
print(counts)

In [ ]:
# Key numeric feature distributions
key_features = ['Credit_Score', 'Debt_to_income', 'CLoan_to_value', 'OLoan_to_value', 'Mortgage_Insurance']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(key_features):
    ax = axes[i//3, i%3]
    sns.histplot(train[col], kde=True, ax=ax)
    ax.set_title(col)
axes[1,2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Credit score by default status
plt.figure(figsize=(6,5))
sns.boxplot(x='DFlag', y='Credit_Score', data=train)
plt.title("Credit Score by Default Status")
plt.show()
print(train.groupby('DFlag')['Credit_Score'].describe())

In [ ]:
# Correlation heatmap
corr_cols = key_features + ['DFlag']
plt.figure(figsize=(8,6))
sns.heatmap(train[corr_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap of Key Features")
plt.show()

In [ ]:
# Outlier check
print("\nOutlier check (IQR method):")
for col in key_features:
    q1, q3 = train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers = train[(train[col] < lower) | (train[col] > upper)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(train)*100:.2f}%)")

**Preprocessing**

In [ ]:
# Drop identifier column
for d in [train, oos, oot]:
    d.drop(columns=['Loanref'], inplace=True, errors='ignore')

In [ ]:
# Drop rows with missing values
train = train.dropna()
oos = oos.dropna()
oot = oot.dropna()

X_train, y_train = train.drop(columns=['DFlag']), train['DFlag']
X_oos, y_oos = oos.drop(columns=['DFlag']), oos['DFlag']
X_oot, y_oot = oot.drop(columns=['DFlag']), oot['DFlag']

In [ ]:
# Scaling for Logistic Regression and SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_oos_scaled = scaler.transform(X_oos)
X_oot_scaled = scaler.transform(X_oot)

print("Preprocessing done:", X_train.shape, X_oos.shape, X_oot.shape)

**Feature engineering**

In [ ]:
# Feature engineering
for df in [X_train, X_oos, X_oot]:
    df['LTV_gap'] = df['OLoan_to_value'] - df['CLoan_to_value']
    df['DTI_x_LTV'] = df['Debt_to_income'] * df['CLoan_to_value']
    df['high_risk_flag'] = ((df['Credit_Score'] < 650) & (df['Debt_to_income'] > 40)).astype(int)

print(X_train[['LTV_gap','DTI_x_LTV','high_risk_flag']].describe())

# Re-scale AFTER adding new features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_oos_scaled = scaler.transform(X_oos)
X_oot_scaled = scaler.transform(X_oot)

**Training the models**

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest": BalancedRandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
                              eval_metric='logloss', random_state=42, n_jobs=-1),
    "SVM": SVC(probability=True, class_weight='balanced', kernel='rbf', random_state=42)
}

trained = {}

In [ ]:
svm_sample_idx = train.groupby('DFlag', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 15000), random_state=42)).index

for name, model in models.items():
    print(f"Training {name}...")
    if name == "SVM":
        mask = X_train.index.isin(svm_sample_idx)
        model.fit(X_train_scaled[mask], y_train[mask])
    elif name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
    else:
        model.fit(X_train, y_train)
    trained[name] = model
    print(f"{name} done.")

**Evaluate on OOS test set**

In [ ]:
results = []
for name, model in trained.items():
    X_eval = X_oos_scaled if name in ["Logistic Regression", "SVM"] else X_oos
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]

    results.append({
        "Model": name,
        "Precision": precision_score(y_oos, y_pred),
        "Recall": recall_score(y_oos, y_pred),
        "F1": f1_score(y_oos, y_pred),
        "ROC-AUC": roc_auc_score(y_oos, y_prob)
    })

    cm = confusion_matrix(y_oos, y_pred)
    ConfusionMatrixDisplay(cm).plot()
    plt.title(f"Confusion Matrix of {name}")
    plt.show()

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
print(results_df)

**Threshold tuning**

In [ ]:
from sklearn.metrics import precision_recall_curve

best_name = results_df.iloc[0]["Model"]
best_model = trained[best_name]
X_eval = X_oos_scaled if best_name in ["Logistic Regression", "SVM"] else X_oos

y_prob = best_model.predict_proba(X_eval)[:, 1]
prec, rec, thresholds = precision_recall_curve(y_oos, y_prob)
f1_scores = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-10)
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f"Best model: {best_name}")
print(f"Tuned threshold: {best_threshold:.3f}")
print(f"Tuned Precision: {prec[best_idx]:.4f}, Recall: {rec[best_idx]:.4f}, F1: {f1_scores[best_idx]:.4f}")

y_pred_tuned = (y_prob >= best_threshold).astype(int)
print("Confusion matrix at tuned threshold:")
print(confusion_matrix(y_oos, y_pred_tuned))

**SHAP and LIME on Random Forest**

In [ ]:
import shap

lr_model = trained["Logistic Regression"]

sample = X_oos.sample(2000, random_state=42)
sample_scaled = scaler.transform(sample)

explainer = shap.LinearExplainer(lr_model, X_train_scaled)
shap_values = explainer.shap_values(sample_scaled)

shap.summary_plot(shap_values, sample, show=False)
plt.show()

shap.summary_plot(shap_values, sample, plot_type="bar", show=False)
plt.show()

In [ ]:
import lime
import lime.lime_tabular

lr_model = trained["Logistic Regression"]

X_lime = X_oos.reset_index(drop=True)
X_lime_scaled = scaler.transform(X_lime)

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train_scaled,
    feature_names=X_train.columns.tolist(),
    class_names=["No Default", "Default"],
    mode="classification"
)

preds = lr_model.predict(X_lime_scaled)
idx = np.where(preds == 1)[0][0]

exp = lime_explainer.explain_instance(
    X_lime_scaled[idx],
    lr_model.predict_proba,
    num_features=10
)

print(exp.as_list())

In [ ]:
fig = exp.as_pyplot_figure()
plt.title("LIME Explanation: Local Prediction (Logistic Regression)")
plt.tight_layout()
plt.show()

**PSI drift monitoring on OOT**

In [ ]:
from scipy.stats import ks_2samp


# PSI + KS Drift Monitoring


def calculate_psi(expected, actual, buckets=10):
    breakpoints = np.linspace(0, 100, buckets + 1)
    bp_values = np.percentile(expected, breakpoints)
    bp_values[0], bp_values[-1] = -np.inf, np.inf

    expected_pct = np.histogram(expected, bp_values)[0] / len(expected)
    actual_pct = np.histogram(actual, bp_values)[0] / len(actual)

    expected_pct = np.where(expected_pct == 0, 0.0001, expected_pct)
    actual_pct = np.where(actual_pct == 0, 0.0001, actual_pct)

    return np.sum(
        (actual_pct - expected_pct) *
        np.log(actual_pct / expected_pct)
    )


drift_results = []

for col in X_train.columns:

    train_values = X_train[col].values
    oot_values = X_oot[col].values

    psi_val = calculate_psi(train_values, oot_values)
    ks_stat, ks_pvalue = ks_2samp(train_values, oot_values)

    drift_results.append({
        "Feature": col,
        "PSI": psi_val,
        "KS_Statistic": ks_stat,
        "KS_p_value": ks_pvalue
    })

drift_df = pd.DataFrame(drift_results)

# PSI interpretation
drift_df["PSI_Flag"] = drift_df["PSI"].apply(
    lambda x: "Significant drift" if x > 0.25
    else ("Moderate drift" if x > 0.10 else "No drift")
)

# KS interpretation
drift_df["KS_Flag"] = drift_df["KS_Statistic"].apply(
    lambda x: "High drift" if x > 0.10
    else ("Moderate drift" if x > 0.05 else "Low drift")
)

drift_df = drift_df.sort_values(
    ["PSI", "KS_Statistic"],
    ascending=False
)

display(drift_df.head(15))

drift_df.to_csv("drift_monitoring_results.csv", index=False)


# -------------------------------
# OOT Performance
# -------------------------------

best_model = trained[best_name]

X_oot_eval = (
    X_oot_scaled
    if best_name in ["Logistic Regression", "SVM"]
    else X_oot
)

y_pred_oot = best_model.predict(X_oot_eval)
y_prob_oot = best_model.predict_proba(X_oot_eval)[:, 1]

print(f"\nOOT Performance ({best_name}):")
print("Precision:", precision_score(y_oot, y_pred_oot))
print("Recall:", recall_score(y_oot, y_pred_oot))
print("F1:", f1_score(y_oot, y_pred_oot))
print("ROC-AUC:", roc_auc_score(y_oot, y_prob_oot))

In [ ]:
# Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import loguniform

cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

param_grids = {
    "Logistic Regression": {
        "C": loguniform(0.01, 10)
    },

    "Random Forest": {
        "n_estimators": [100, 150],
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 5]
    },

    "XGBoost": {
        "n_estimators": [100, 150],
        "max_depth": [3, 5],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.8, 1.0]
    },

    "SVM": {
        "C": [0.1, 1, 10],
        "gamma": ["scale", 0.01],
        "kernel": ["rbf"]
    }
}

In [ ]:
tuned_models = {}

# Smaller stratified sample for SVM
svm_idx = train.groupby(
    "DFlag",
    group_keys=False
).apply(
    lambda x: x.sample(min(len(x), 5000), random_state=42)
).index

for name, model in models.items():

    print(f"Tuning {name}...")

    if name == "SVM":
        X_tune = X_train_scaled[X_train.index.isin(svm_idx)]
        y_tune = y_train.loc[svm_idx]
    elif name == "Logistic Regression":
        X_tune = X_train_scaled
        y_tune = y_train
    else:
        X_tune = X_train
        y_tune = y_train

    search = RandomizedSearchCV(
        model,
        param_distributions=param_grids[name],
        n_iter=3,
        scoring="f1",
        cv=cv,
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_tune, y_tune)

    tuned_models[name] = search.best_estimator_

    print("Best parameters:", search.best_params_)
    print("Best CV F1:", round(search.best_score_, 4))

In [ ]:
trained = tuned_models
print("Tuning complete.")

In [ ]:
# SMOTE for class balancing

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Use Logistic Regression for the imbalance comparison
# Training data only is resampled; OOS data remains untouched.

X_train_lr = X_train_scaled
X_oos_lr = X_oos_scaled

models_imbalance = {}

# 1. Cost-sensitive learning (existing approach)
models_imbalance["Cost-sensitive"] = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# 2. SMOTE
smote = SMOTE(
    sampling_strategy=0.1,
    random_state=42
)

X_smote, y_smote = smote.fit_resample(X_train_lr, y_train)

models_imbalance["SMOTE"] = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# 3. Random undersampling
under = RandomUnderSampler(
    sampling_strategy=0.1,
    random_state=42
)

X_under, y_under = under.fit_resample(X_train_lr, y_train)

models_imbalance["Undersampling"] = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train and evaluate
imbalance_results = []

for name, model in models_imbalance.items():

    if name == "SMOTE":
        model.fit(X_smote, y_smote)

    elif name == "Undersampling":
        model.fit(X_under, y_under)

    else:
        model.fit(X_train_lr, y_train)

    y_pred = model.predict(X_oos_lr)
    y_prob = model.predict_proba(X_oos_lr)[:, 1]

    imbalance_results.append({
        "Method": name,
        "Precision": precision_score(y_oos, y_pred, zero_division=0),
        "Recall": recall_score(y_oos, y_pred, zero_division=0),
        "F1": f1_score(y_oos, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_oos, y_prob)
    })

imbalance_results_df = pd.DataFrame(imbalance_results)

display(
    imbalance_results_df.sort_values("F1", ascending=False)
)

In [ ]:
!pip install dash dash-bootstrap-components plotly

**Prediction Dashboard Development**

In [ ]:
import dash
from dash import dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import plotly.express as px

# SETUP MODEL & BASELINE DISTRIBUTIONS

feature_cols = [
    'Credit_Score', 'Mortgage_Insurance', 'Number_of_units',
    'CLoan_to_value', 'Debt_to_income', 'OLoan_to_value', 'Single_borrower'
]

np.random.seed(42)
n_samples = 2000
demo_X = pd.DataFrame({
    'Credit_Score': np.random.uniform(300, 850, n_samples),
    'Mortgage_Insurance': np.random.uniform(0, 50, n_samples),
    'Number_of_units': np.random.choice([1, 2, 3, 4], n_samples),
    'CLoan_to_value': np.random.uniform(10, 120, n_samples),
    'Debt_to_income': np.random.uniform(1, 60, n_samples),
    'OLoan_to_value': np.random.uniform(10, 120, n_samples),
    'Single_borrower': np.random.choice([0, 1], n_samples)
})

baseline_means = demo_X.mean()
baseline_stds = demo_X.std()

risk_score_formula = (
    (850 - demo_X['Credit_Score']) * 0.4 +
    demo_X['Debt_to_income'] * 1.5 +
    demo_X['CLoan_to_value'] * 1.2 +
    demo_X['OLoan_to_value'] * 0.8 +
    demo_X['Single_borrower'] * 10
)
demo_y = (risk_score_formula > np.percentile(risk_score_formula, 60)).astype(int)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(demo_X, demo_y)


# STYLES & LAYOUT

CARD_STYLE = {
    "backgroundColor": "#1e293b",
    "borderColor": "#334155",
    "color": "#f8fafc"
}

CARD_HEADER_STYLE = {
    "backgroundColor": "#1e293b",
    "borderColor": "#334155",
    "color": "#f8fafc"
}

MARK_STYLE = {"color": "#ffffff", "fontWeight": "bold", "fontSize": "0.85rem"}

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])

app.layout = dbc.Container([
    dcc.Markdown("""
        <style>
            .form-check-label {
                color: #f8fafc !important;
                font-weight: 600 !important;
            }
            input[type=number] {
                color: #ffffff !important;
                background-color: #0f172a !important;
                border: 1px solid #334155 !important;
                font-weight: 700 !important;
            }
        </style>
    """, dangerously_allow_html=True),

    # Header
    dbc.Row([
        dbc.Col([
            html.H5("CREDIT RISK SCORE ASSESSMENT SYSTEM",
                    className="text-white py-2 px-3 rounded text-center my-1 font-weight-bold",
                    style={"fontSize": "1.2rem", "backgroundColor": "#0284c7"})
        ], width=12)
    ], className="mb-2"),

    dbc.Row([
        # Left Panel (Inputs)
        dbc.Col([
            dbc.Card([
                dbc.CardHeader(html.H6("Applicant Profile Inputs", className="mb-0 font-weight-bold", style={"fontSize": "0.95rem", "color": "#f8fafc"}), style=CARD_HEADER_STYLE),
                dbc.CardBody([

                    # Credit Score Slider
                    html.Div([
                        html.Div([
                            html.Small("Credit Score: ", className="font-weight-bold", style={"color": "#f8fafc", "fontSize": "0.85rem"}),
                            dbc.Badge(id="credit-score-badge", color="info", className="ms-2", style={"fontSize": "0.85rem", "backgroundColor": "#0284c7"})
                        ], className="d-flex justify-content-between align-items-center mb-1"),
                        dcc.Slider(
                            id='credit-score', min=300, max=850, step=1, value=795,
                            marks={
                                300: {'label': '300', 'style': MARK_STYLE},
                                580: {'label': '580', 'style': MARK_STYLE},
                                740: {'label': '740', 'style': MARK_STYLE},
                                850: {'label': '850', 'style': MARK_STYLE}
                            },
                            tooltip={"always_visible": False}
                        )
                    ], className="mb-3"),

                    # Debt-to-Income Slider
                    html.Div([
                        html.Div([
                            html.Small("Debt-to-Income Ratio (%): ", className="font-weight-bold", style={"color": "#f8fafc", "fontSize": "0.85rem"}),
                            dbc.Badge(id="dti-badge", color="info", className="ms-2", style={"fontSize": "0.85rem", "backgroundColor": "#0284c7"})
                        ], className="d-flex justify-content-between align-items-center mb-1"),
                        dcc.Slider(
                            id='dti', min=1, max=60, step=1, value=46,
                            marks={
                                10: {'label': '10%', 'style': MARK_STYLE},
                                25: {'label': '25%', 'style': MARK_STYLE},
                                50: {'label': '50%', 'style': MARK_STYLE}
                            },
                            tooltip={"always_visible": False}
                        )
                    ], className="mb-3"),

                    # CLTV Slider
                    html.Div([
                        html.Div([
                            html.Small("Combined Loan-to-Value (CLTV %): ", className="font-weight-bold", style={"color": "#f8fafc", "fontSize": "0.85rem"}),
                            dbc.Badge(id="cltv-badge", color="info", className="ms-2", style={"fontSize": "0.85rem", "backgroundColor": "#0284c7"})
                        ], className="d-flex justify-content-between align-items-center mb-1"),
                        dcc.Slider(
                            id='cltv', min=10, max=120, step=1, value=98,
                            marks={
                                20: {'label': '20%', 'style': MARK_STYLE},
                                80: {'label': '80%', 'style': MARK_STYLE},
                                120: {'label': '120%', 'style': MARK_STYLE}
                            },
                            tooltip={"always_visible": False}
                        )
                    ], className="mb-3"),

                    # LTV Slider
                    html.Div([
                        html.Div([
                            html.Small("Original Loan-to-Value (LTV %): ", className="font-weight-bold", style={"color": "#f8fafc", "fontSize": "0.85rem"}),
                            dbc.Badge(id="ltv-badge", color="info", className="ms-2", style={"fontSize": "0.85rem", "backgroundColor": "#0284c7"})
                        ], className="d-flex justify-content-between align-items-center mb-1"),
                        dcc.Slider(
                            id='ltv', min=10, max=120, step=1, value=94,
                            marks={
                                20: {'label': '20%', 'style': MARK_STYLE},
                                80: {'label': '80%', 'style': MARK_STYLE},
                                120: {'label': '120%', 'style': MARK_STYLE}
                            },
                            tooltip={"always_visible": False}
                        )
                    ], className="mb-3"),

                    dbc.Row([
                        dbc.Col([
                            html.Small("Mortgage Insurance (%):", className="font-weight-bold", style={"color": "#f8fafc", "fontSize": "0.85rem"}),
                            dbc.Input(id='mi', type='number', min=0, max=50, value=0, size="sm", className="mb-2")
                        ], width=6),
                        dbc.Col([
                            html.Small("Units:", className="font-weight-bold", style={"color": "#f8fafc", "fontSize": "0.85rem"}),
                            dbc.RadioItems(
                                id='units',
                                options=[{'label': f'{i}', 'value': i} for i in range(1, 5)],
                                value=1, inline=True, style={"fontSize": "0.85rem"}
                            )
                        ], width=6)
                    ], className="mb-2"),

                    html.Div([
                        html.Small("Borrower Account Profile:", className="font-weight-bold", style={"color": "#f8fafc", "fontSize": "0.85rem"}),
                        dbc.RadioItems(
                            id='single-borrower',
                            options=[{'label': 'Single', 'value': 1}, {'label': 'Joint', 'value': 0}],
                            value=1, inline=True, style={"fontSize": "0.85rem"}, className="mb-3"
                        )
                    ]),

                    dbc.Button("Predict Default Risk", id="predict-btn", color="danger", size="md", className="w-100 font-weight-bold mt-2")
                ], style={"padding": "16px"})
            ], className="shadow-sm h-100", style=CARD_STYLE)
        ], width=4),

        # Right Panel (Analytics Dashboard)
        dbc.Col([
            # Top Executive Summary KPIs
            dbc.Row([
                dbc.Col([
                    dbc.Card([
                        dbc.CardBody([
                            html.Div("Default Risk Score", className="text-center", style={"fontSize": "0.75rem", "color": "#94a3b8"}),
                            html.H5(id="risk-score-text", className="text-center font-weight-bold mb-0", style={"fontSize": "1.1rem"})
                        ], style={"padding": "10px"})
                    ], className="shadow-sm mb-3", style=CARD_STYLE)
                ], style={"width": "20%", "flex": "0 0 20%", "maxWidth": "20%"}),

                dbc.Col([
                    dbc.Card([
                        dbc.CardBody([
                            html.Div("Risk Classification", className="text-center", style={"fontSize": "0.75rem", "color": "#94a3b8"}),
                            html.H5(id="risk-level-text", className="text-center font-weight-bold mb-0", style={"fontSize": "1.0rem"})
                        ], style={"padding": "10px"})
                    ], className="shadow-sm mb-3", style=CARD_STYLE)
                ], style={"width": "20%", "flex": "0 0 20%", "maxWidth": "20%"}),

                dbc.Col([
                    dbc.Card([
                        dbc.CardBody([
                            html.Div("Credit Approval Status", className="text-center", style={"fontSize": "0.75rem", "color": "#94a3b8"}),
                            html.H5(id="decision-text", className="text-center font-weight-bold mb-0", style={"fontSize": "1.0rem"})
                        ], style={"padding": "10px"})
                    ], className="shadow-sm mb-3", style=CARD_STYLE)
                ], style={"width": "20%", "flex": "0 0 20%", "maxWidth": "20%"}),

                dbc.Col([
                    dbc.Card([
                        dbc.CardBody([
                            html.Div("Risk Premium (+bps)", className="text-center", style={"fontSize": "0.75rem", "color": "#94a3b8"}),
                            html.H5(id="premium-text", className="text-center font-weight-bold mb-0", style={"fontSize": "1.1rem"})
                        ], style={"padding": "10px"})
                    ], className="shadow-sm mb-3", style=CARD_STYLE)
                ], style={"width": "20%", "flex": "0 0 20%", "maxWidth": "20%"}),

                dbc.Col([
                    dbc.Card([
                        dbc.CardBody([
                            html.Div("Drift Score", className="text-center", style={"fontSize": "0.75rem", "color": "#94a3b8"}),
                            html.H5(id="drift-score-text", className="text-center font-weight-bold mb-0", style={"fontSize": "1.1rem"})
                        ], style={"padding": "10px"})
                    ], className="shadow-sm mb-3", style=CARD_STYLE)
                ], style={"width": "20%", "flex": "0 0 20%", "maxWidth": "20%"}),
            ], className="no-gutters"),

            # Middle Row
            dbc.Row([
                dbc.Col([
                    dbc.Card([
                        dbc.CardHeader(html.Div("Probability Gauge", className="font-weight-bold", style={"fontSize": "0.85rem", "color": "#f8fafc"}), style=CARD_HEADER_STYLE),
                        dbc.CardBody([dcc.Graph(id="gauge-chart", style={"height": "210px"})], style={"padding": "2px"})
                    ], className="shadow-sm mb-3", style=CARD_STYLE)
                ], width=4),

                dbc.Col([
                    dbc.Card([
                        dbc.CardHeader(html.Div("💡 Risk Mitigation Recommendations", className="font-weight-bold", style={"fontSize": "0.85rem", "color": "#f8fafc"}), style=CARD_HEADER_STYLE),
                        dbc.CardBody([
                            html.Ul(id="suggestions-list", className="pl-3 mb-0", style={"fontSize": "0.82rem", "lineHeight": "1.4", "color": "#cbd5e1"})
                        ], style={"padding": "12px", "height": "210px", "overflowY": "auto"})
                    ], className="shadow-sm mb-3", style={**CARD_STYLE, "borderLeft": "4px solid #38bdf8"})
                ], width=4),

                dbc.Col([
                    dbc.Card([
                        dbc.CardHeader(html.Div("Applicant Risk Factors", className="font-weight-bold", style={"fontSize": "0.85rem", "color": "#f8fafc"}), style=CARD_HEADER_STYLE),
                        dbc.CardBody([dcc.Graph(id="waterfall-chart", style={"height": "210px"})], style={"padding": "2px"})
                    ], className="shadow-sm mb-3", style=CARD_STYLE)
                ], width=4)
            ]),

            # Bottom Row
            dbc.Row([
                dbc.Col([
                    dbc.Card([
                        dbc.CardHeader(html.Div("Peer Benchmark Comparison", className="font-weight-bold", style={"fontSize": "0.85rem", "color": "#f8fafc"}), style=CARD_HEADER_STYLE),
                        dbc.CardBody([dcc.Graph(id="benchmark-chart", style={"height": "210px"})], style={"padding": "2px"})
                    ], className="shadow-sm", style=CARD_STYLE)
                ], width=6),
                dbc.Col([
                    dbc.Card([
                        dbc.CardHeader(html.Div("Global Feature Importance", className="font-weight-bold", style={"fontSize": "0.85rem", "color": "#f8fafc"}), style=CARD_HEADER_STYLE),
                        dbc.CardBody([dcc.Graph(id="feature-importance-chart", style={"height": "210px"})], style={"padding": "2px"})
                    ], className="shadow-sm", style=CARD_STYLE)
                ], width=6)
            ])
        ], width=8)
    ], className="flex-grow-1")
], fluid=True, style={"minHeight": "100vh", "display": "flex", "flexDirection": "column", "paddingBottom": "15px", "backgroundColor": "#0f172a"})


# CALLBACK TO UPDATE SLIDER BADGES DYNAMICALLY

@app.callback(
    [Output("credit-score-badge", "children"),
     Output("dti-badge", "children"),
     Output("cltv-badge", "children"),
     Output("ltv-badge", "children")],
    [Input("credit-score", "value"),
     Input("dti", "value"),
     Input("cltv", "value"),
     Input("ltv", "value")]
)
def update_slider_badges(cs, dti, cltv, ltv):
    return f"{cs}", f"{dti}%", f"{cltv}%", f"{ltv}%"


# CALLBACK TO UPDATE PREDICTION ANALYTICS

@app.callback(
    [Output("risk-score-text", "children"), Output("risk-score-text", "style"),
     Output("risk-level-text", "children"), Output("risk-level-text", "style"),
     Output("decision-text", "children"), Output("decision-text", "style"),
     Output("premium-text", "children"), Output("premium-text", "style"),
     Output("drift-score-text", "children"), Output("drift-score-text", "style"),
     Output("suggestions-list", "children"),
     Output("gauge-chart", "figure"),
     Output("waterfall-chart", "figure"),
     Output("benchmark-chart", "figure"),
     Output("feature-importance-chart", "figure")],
    [Input("predict-btn", "n_clicks")],
    [State("credit-score", "value"),
     State("mi", "value"),
     State("units", "value"),
     State("cltv", "value"),
     State("dti", "value"),
     State("ltv", "value"),
     State("single-borrower", "value")]
)
def update_prediction(n_clicks, credit_score, mi, units, cltv, dti, ltv, single_borrower):
    mi = mi if mi is not None else 0

    input_data = pd.DataFrame([[
        credit_score, mi, units, cltv, dti, ltv, single_borrower
    ]], columns=feature_cols)

    prob = model.predict_proba(input_data)[0][1]
    raw_percentage = prob * 100
    formatted_score = f"{raw_percentage:.2f}%"

    z_scores = np.abs((input_data.iloc[0] - baseline_means) / baseline_stds)
    drift_score = np.mean(z_scores) * 0.05

    if drift_score < 0.10:
        drift_text = f"{drift_score:.2f} (LOW)"
        drift_color = "#4ade80"
    elif drift_score < 0.25:
        drift_text = f"{drift_score:.2f} (MOD)"
        drift_color = "#fbbf24"
    else:
        drift_text = f"{drift_score:.2f} (HIGH)"
        drift_color = "#f87171"

    if raw_percentage < 25.0:
        risk_level = "LOW RISK"
        decision = "APPROVE"
        premium = "+0 bps"
        color = "#4ade80"
    elif raw_percentage < 60.0:
        risk_level = "MODERATE RISK"
        decision = "REVIEW"
        premium = "+75 bps"
        color = "#fbbf24"
    else:
        risk_level = "HIGH RISK"
        decision = "DECLINE"
        premium = "+250 bps"
        color = "#f87171"

    suggestions = []
    if dti > 40:
        suggestions.append(html.Li([html.B("Lower DTI: "), f"Reduce DTI from {dti}% to < 40% (pay down short-term debt)."]))
    if cltv > 80:
        target_down = round((cltv - 80) * 0.5, 1)
        suggestions.append(html.Li([html.B("Increase Down Payment: "), f"Lower CLTV ({cltv}%) to 80% (add ~{target_down}% down payment)."]))
    if credit_score < 680:
        suggestions.append(html.Li([html.B("Credit Score: "), "Improve credit score to 700+ to lower risk bracket."]))
    if single_borrower == 1:
        suggestions.append(html.Li([html.B("Co-Signer: "), "Add a co-borrower to share liability."]))
    if mi == 0 and (ltv > 80 or cltv > 80):
        suggestions.append(html.Li([html.B("Mortgage Insurance: "), "Add PMI coverage to offset high LTV risk."]))

    if not suggestions:
        suggestions.append(html.Li("Applicant meets optimal low-risk criteria. No immediate changes needed.", style={"color": "#4ade80"}))

    dark_layout = dict(
        paper_bgcolor="#1e293b",
        plot_bgcolor="#1e293b",
        font=dict(color="#f8fafc")
    )

    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=round(raw_percentage, 2),
        number={'suffix': "%", 'font': {'size': 20, 'color': '#f8fafc'}},
        gauge={
            'axis': {'range': [0, 100], 'tickfont': {'size': 10, 'color': '#f8fafc'}},
            'bar': {'color': color},
            'steps': [
                {'range': [0, 25], 'color': "#064e3b"},
                {'range': [25, 60], 'color': "#78350f"},
                {'range': [60, 100], 'color': "#7f1d1d"}
            ]
        }
    ))
    fig_gauge.update_layout(**dark_layout, margin=dict(l=15, r=15, t=15, b=15))

    cs_impact = (700 - credit_score) * 0.08
    dti_impact = (dti - 35) * 0.3
    cltv_impact = (cltv - 75) * 0.2

    fig_waterfall = go.Figure(go.Waterfall(
        orientation="v",
        measure=["relative", "relative", "relative", "total"],
        x=["CS", "DTI", "CLTV", "Final"],
        textposition="none",
        y=[cs_impact, dti_impact, cltv_impact, None],
        connector={"line": {"color": "#64748b"}},
        decreasing={"marker": {"color": "#4ade80"}},
        increasing={"marker": {"color": "#f87171"}},
        totals={"marker": {"color": color}}
    ))
    fig_waterfall.update_layout(**dark_layout, margin=dict(l=10, r=10, t=10, b=10), showlegend=False, xaxis={'tickfont': {'size': 10, 'color': '#f8fafc'}}, yaxis={'tickfont': {'size': 10, 'color': '#f8fafc'}})

    fig_benchmark = go.Figure(data=[
        go.Bar(name='Applicant', x=['Credit Score', 'DTI Ratio', 'CLTV Ratio'], y=[credit_score, dti, cltv], marker_color=color),
        go.Bar(name='Industry Avg', x=['Credit Score', 'DTI Ratio', 'CLTV Ratio'], y=[720, 36, 75], marker_color='#64748b')
    ])
    fig_benchmark.update_layout(**dark_layout, barmode='group', margin=dict(l=10, r=10, t=10, b=10), showlegend=True, legend=dict(font=dict(size=9, color="#f8fafc"), orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1), xaxis={'tickfont': {'size': 10, 'color': '#f8fafc'}}, yaxis={'tickfont': {'size': 10, 'color': '#f8fafc'}})

    importances = model.feature_importances_
    feat_df = pd.DataFrame({'Feature': feature_cols, 'Importance': importances}).sort_values(by='Importance', ascending=True)

    fig_importance = px.bar(
        feat_df, x='Importance', y='Feature', orientation='h',
        color='Importance', color_continuous_scale='Reds'
    )
    fig_importance.update_layout(
        **dark_layout,
        showlegend=False,
        xaxis={'tickfont': {'size': 10, 'color': '#f8fafc'}, 'title': None},
        yaxis={'tickfont': {'size': 10, 'color': '#f8fafc'}, 'title': None},
        margin=dict(l=10, r=10, t=10, b=10),
        coloraxis_showscale=False
    )

    return (
        formatted_score, {"color": color},
        risk_level, {"color": color},
        decision, {"color": color},
        premium, {"color": color},
        drift_text, {"color": drift_color},
        suggestions,
        fig_gauge,
        fig_waterfall,
        fig_benchmark,
        fig_importance
    )

if __name__ == "__main__":
    app.run(jupyter_mode="external", port=8050)